# Projet : Étude de marché internationale pour le poulet biologique

## Présentation du projet

Ce projet vise à réaliser une **étude de marché internationale** pour une entreprise souhaitant commercialiser du **poulet biologique** à l'échelle mondiale.  
L'analyse repose sur des **données mondiales** portant sur les **habitudes alimentaires** et les **politiques agricoles** de différents pays.

L'objectif principal de ce notebook est de **préparer et structurer les données** en vue de l'étape la plus stratégique de l'analyse :  
**la sélection du ou des pays cibles**, en cohérence avec la **stratégie de l'entreprise** et les **indicateurs de marché les plus pertinents**.

---

## Objectifs du notebook

Ce notebook se concentre exclusivement sur la **phase de préparation des données**, étape indispensable avant toute analyse décisionnelle.

Les étapes suivantes seront réalisées :

### 1. Exploration et préparation des données
- Analyse des fichiers sources  
- Compréhension des structures et des variables  
- Transformation et harmonisation des formats de données  

### 2. Nettoyage des données
- Traitement des valeurs manquantes  
- Suppression des doublons  
- Correction des données mal formatées ou incohérentes  

### 3. Construction du jeu de données final
- Sélection des variables pertinentes pour l'étude de marché  
- Agrégation et regroupement des données au niveau pays  
- Création d'un **jeu de données final propre et exploitable**, prêt pour la suite de l'analyse et la visualisation


In [ ]:
# Import useful libraries
import pandas as pd

In [ ]:
# Import CSV files

path = 'data/'
extension = '.csv'

df_dispo_alimentaire = pd.read_csv(f'{path}DisponibiliteAlimentaire_2017{extension}', sep=',')
df_population = pd.read_csv(f'{path}Population_2000_2018{extension}')

# **Fonctions utiles**

Eviter la redondance de certains traitements comme:
- la visualisation du nombre de valeurs
- le renommage des colonnes

In [ ]:
def rename_columns(df, old_names, new_names):
  """Renomme plusieurs colonnes dans un DataFrame.

  Args:
    df (pd.DataFrame): Le DataFrame d'entrée.
    old_names (list): Une liste d'anciens noms de colonnes.
    new_names (list): Une liste de nouveaux noms de colonnes.

  Returns:
    pd.DataFrame: Le DataFrame avec les colonnes renommées.
  """
  for old_name, new_name in zip(old_names, new_names):
    df = df.rename(columns={old_name: new_name})
  return df

def rename_column(df, old_name, new_name):
  """Renomme une seule colonne dans un DataFrame.

  Args:
    df (pd.DataFrame): Le DataFrame d'entrée.
    old_name (str): L'ancien nom de la colonne.
    new_name (str): Le nouveau nom de la colonne.

  Returns:
    pd.DataFrame: Le DataFrame avec la colonne renommée.
  """
  df = df.rename(columns={old_name: new_name})
  return df

In [ ]:
def show_columns(df):
  """Affiche le décompte des valeurs pour chaque colonne d'un DataFrame.

  Args:
    df (pd.DataFrame): Le DataFrame d'entrée.
  """
  for col in df.columns:
    print(df[col].value_counts(dropna=False))

# **Analyse exploratoire du dataframe DISPO ALIMENTAIRE**

Vérification de la structure de la table: type de données, nombre de lignes et le nombre de valeurs non-nulles et le nombre de lignes max.

In [ ]:
df_dispo_alimentaire.info()

show_columns(df_dispo_alimentaire)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 176600 entries, 0 to 176599
Data columns (total 14 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Code Domaine            176600 non-null  object 
 1   Domaine                 176600 non-null  object 
 2   Code zone               176600 non-null  int64  
 3   Zone                    176600 non-null  object 
 4   Code Élément            176600 non-null  int64  
 5   Élément                 176600 non-null  object 
 6   Code Produit            176600 non-null  int64  
 7   Produit                 176600 non-null  object 
 8   Code année              176600 non-null  int64  
 9   Année                   176600 non-null  int64  
 10  Unité                   176600 non-null  object 
 11  Valeur                  176600 non-null  float64
 12  Symbole                 176600 non-null  object 
 13  Description du Symbole  176600 non-null  object 
dtypes: float64(1), int64

In [ ]:
# Keep only Zone, Element, Année, Valeur usefull for the analysis
df_dispo_alimentaire = df_dispo_alimentaire[['Zone', 'Élément', 'Année', 'Valeur', 'Produit', 'Unité']]

df_dispo_alimentaire.info()

df_dispo_alimentaire.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 176600 entries, 0 to 176599
Data columns (total 6 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   Zone     176600 non-null  object 
 1   Élément  176600 non-null  object 
 2   Année    176600 non-null  int64  
 3   Valeur   176600 non-null  float64
 4   Produit  176600 non-null  object 
 5   Unité    176600 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 8.1+ MB


,Zone,Élément,Année,Valeur,Produit,Unité
0,Afghanistan,Production,2017,4281.0,Blé et produits,Milliers de tonnes
1,Afghanistan,Importations - Quantité,2017,2302.0,Blé et produits,Milliers de tonnes
2,Afghanistan,Variation de stock,2017,-119.0,Blé et produits,Milliers de tonnes
3,Afghanistan,Exportations - Quantité,2017,0.0,Blé et produits,Milliers de tonnes
4,Afghanistan,Disponibilité intérieure,2017,6701.0,Blé et produits,Milliers de tonnes


Après visualisation, une transformation profonde de la table est nécessaire pour éviter la redondances des informations (Année, Unité et Produit) et, mettre en avant le champs "élément" qui contient les éléments qui vont servir de base à l'étude de marché

In [ ]:
# Transpose the dataframe to have it indexed by the Zone and Produit
df_dispo_alimentaire_pivot = df_dispo_alimentaire.pivot_table(index=['Zone', 'Produit', 'Année'], columns='Élément', values='Valeur')

df_dispo_alimentaire_pivot.head()

# Check that all the values are in the right format
df_dispo_alimentaire_pivot.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 16047 entries, ('Afghanistan', 'Abats Comestible', np.int64(2017)) to ('Îles Salomon', 'Épices, Autres', np.int64(2017))
Data columns (total 17 columns):
 #   Column                                                         Non-Null Count  Dtype  
---  ------                                                         --------------  -----  
 0   Alimentation pour touristes                                    5560 non-null   float64
 1   Aliments pour animaux                                          4188 non-null   float64
 2   Autres utilisations (non alimentaire)                          5292 non-null   float64
 3   Disponibilité alimentaire (Kcal/personne/jour)                 14476 non-null  float64
 4   Disponibilité alimentaire en quantité (kg/personne/an)         14618 non-null  float64
 5   Disponibilité de matière grasse en quantité (g/personne/jour)  14512 non-null  float64
 6   Disponibilité de protéines en quantité (g/personne/jour

In [ ]:
# Replace all the NaN values by 0
df_dispo_alimentaire_pivot = df_dispo_alimentaire_pivot.fillna(0)

# Reset the index
df_dispo_alimentaire_pivot = df_dispo_alimentaire_pivot.reset_index()

Toutes les valeurs nulles (NaN) ont été remplacées par 0 par soucis de simplification. Une solution plus adaptée à l'exclusion des lignes concernées, car cela pourrait réduire le nombre de données et de pays pour la pertinence de l'étude.

In [ ]:
# Check if there any duplicates
duplicated = df_dispo_alimentaire_pivot[['Zone', 'Produit', 'Année']].duplicated().sum()
print(f'There are {duplicated} duplicates in the dataframe')

df_dispo_alimentaire_pivot.head()

There are 0 duplicates in the dataframe


Élément,Zone,Produit,Année,Alimentation pour touristes,Aliments pour animaux,Autres utilisations (non alimentaire),Disponibilité alimentaire (Kcal/personne/jour),Disponibilité alimentaire en quantité (kg/personne/an),Disponibilité de matière grasse en quantité (g/personne/jour),Disponibilité de protéines en quantité (g/personne/jour),Disponibilité intérieure,Exportations - Quantité,Importations - Quantité,Nourriture,Pertes,Production,Résidus,Semences,Traitement,Variation de stock
0,Afghanistan,Abats Comestible,2017,0.0,0.0,0.0,5.0,1.47,0.19,0.64,53.0,0.0,6.0,53.0,0.0,48.0,0.0,0.0,0.0,0.0
1,Afghanistan,"Agrumes, Autres",2017,0.0,0.0,0.0,1.0,1.32,0.01,0.02,50.0,0.0,33.0,48.0,2.0,17.0,0.0,0.0,0.0,0.0
2,Afghanistan,"Alcool, non Comestible",2017,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Afghanistan,Aliments pour enfants,2017,0.0,0.0,0.0,1.0,0.10,0.01,0.04,4.0,0.0,4.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Afghanistan,Ananas et produits,2017,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Rename the column Zone
df_dispo_alimentaire_pivot = rename_column(df_dispo_alimentaire_pivot, 'Zone', 'Pays')
df_dispo_alimentaire_pivot.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16047 entries, 0 to 16046
Data columns (total 20 columns):
 #   Column                                                         Non-Null Count  Dtype  
---  ------                                                         --------------  -----  
 0   Pays                                                           16047 non-null  object 
 1   Produit                                                        16047 non-null  object 
 2   Année                                                          16047 non-null  int64  
 3   Alimentation pour touristes                                    16047 non-null  float64
 4   Aliments pour animaux                                          16047 non-null  float64
 5   Autres utilisations (non alimentaire)                          16047 non-null  float64
 6   Disponibilité alimentaire (Kcal/personne/jour)                 16047 non-null  float64
 7   Disponibilité alimentaire en quantité (kg/personne/an)    

# **Analyse exploratoire du dataframe POPULATION**

Vérifier s'il existe des incocohérences dans le formattage des données. Une ville par exemple ou tout autre label de type catégorie, peut être redondant selon son orthographe.

In [ ]:
show_columns(df_population)

Code Domaine
OA    4411
Name: count, dtype: int64
Domaine
Séries temporelles annuelles    4411
Name: count, dtype: int64
Code zone
2      19
202    19
3      19
4      19
79     19
       ..
281     8
280     8
276     7
277     7
186     6
Name: count, Length: 238, dtype: int64
Zone
Afghanistan                            19
Afrique du Sud                         19
Albanie                                19
Algérie                                19
Allemagne                              19
                                       ..
Saint-Martin (partie française)         8
Sint Maarten  (partie néerlandaise)     8
Soudan                                  7
Soudan du Sud                           7
Serbie-et-Monténégro                    6
Name: count, Length: 238, dtype: int64
Code Élément
511    4411
Name: count, dtype: int64
Élément
Population totale    4411
Name: count, dtype: int64
Code Produit
3010    4411
Name: count, dtype: int64
Produit
Population-Estimations    4411
Name: count,

In [ ]:
# See the general outlook of the dataframe
df_population.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4411 entries, 0 to 4410
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Code Domaine            4411 non-null   object 
 1   Domaine                 4411 non-null   object 
 2   Code zone               4411 non-null   int64  
 3   Zone                    4411 non-null   object 
 4   Code Élément            4411 non-null   int64  
 5   Élément                 4411 non-null   object 
 6   Code Produit            4411 non-null   int64  
 7   Produit                 4411 non-null   object 
 8   Code année              4411 non-null   int64  
 9   Année                   4411 non-null   int64  
 10  Unité                   4411 non-null   object 
 11  Valeur                  4411 non-null   float64
 12  Symbole                 4411 non-null   object 
 13  Description du Symbole  4411 non-null   object 
 14  Note                    258 non-null    

In [ ]:
# take the right accurate columns
df_population = df_population[['Zone', 'Année', 'Valeur']]
df_population.head()

,Zone,Année,Valeur
0,Afghanistan,2000,20779.953
1,Afghanistan,2001,21606.988
2,Afghanistan,2002,22600.770
3,Afghanistan,2003,23680.871
4,Afghanistan,2004,24726.684


Pour faciliter l'analyse, certains en-têtes doivent être modifiés pour les rendre explicites (Zone -> Pays, Valeur -> Population totale)

In [ ]:
# Replace Valeur -> Population totale and zone by country
df_population = rename_columns(df_population, ['Valeur', 'Zone'], ['Population totale', 'Pays'])
df_population['Population totale'] = (df_population['Population totale'] * 1000).astype(int)
df_population.head()

,Pays,Année,Population totale
0,Afghanistan,2000,20779953
1,Afghanistan,2001,21606988
2,Afghanistan,2002,22600770
3,Afghanistan,2003,23680871
4,Afghanistan,2004,24726684


In [ ]:
df_population.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4411 entries, 0 to 4410
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Pays               4411 non-null   object
 1   Année              4411 non-null   int64 
 2   Population totale  4411 non-null   int64 
dtypes: int64(2), object(1)
memory usage: 103.5+ KB


# **Fusion et consolidation des fichiers**

Objectif: combiner en une seule table la population pour chaque pays et les métriques à analyser.

In [ ]:
df_population_dispo_alimentaire = pd.merge(df_population, df_dispo_alimentaire_pivot, on=['Pays', 'Année'])
df_population_dispo_alimentaire.head(200)

,Pays,Année,Population totale,Produit,Alimentation pour touristes,Aliments pour animaux,Autres utilisations (non alimentaire),Disponibilité alimentaire (Kcal/personne/jour),Disponibilité alimentaire en quantité (kg/personne/an),Disponibilité de matière grasse en quantité (g/personne/jour),...,Disponibilité intérieure,Exportations - Quantité,Importations - Quantité,Nourriture,Pertes,Production,Résidus,Semences,Traitement,Variation de stock
0,Afghanistan,2017,36296113,Abats Comestible,0.0,0.0,0.0,5.0,1.47,0.19,...,53.0,0.0,6.0,53.0,0.0,48.0,0.0,0.0,0.0,0.0
1,Afghanistan,2017,36296113,"Agrumes, Autres",0.0,0.0,0.0,1.0,1.32,0.01,...,50.0,0.0,33.0,48.0,2.0,17.0,0.0,0.0,0.0,0.0
2,Afghanistan,2017,36296113,"Alcool, non Comestible",0.0,0.0,0.0,0.0,0.00,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Afghanistan,2017,36296113,Aliments pour enfants,0.0,0.0,0.0,1.0,0.10,0.01,...,4.0,0.0,4.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Afghanistan,2017,36296113,Ananas et produits,0.0,0.0,0.0,0.0,0.00,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,Albanie,2017,2884169,Coco (Incl Coprah),0.0,0.0,0.0,0.0,0.00,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
196,Albanie,2017,2884169,Crustacés,0.0,0.0,0.0,0.0,0.35,0.00,...,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
197,Albanie,2017,2884169,Crème,0.0,0.0,0.0,1.0,0.00,0.09,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
198,Albanie,2017,2884169,"Céréales, Autres",0.0,0.0,0.0,1.0,0.05,0.00,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [ ]:
df_population_dispo_alimentaire.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16047 entries, 0 to 16046
Data columns (total 21 columns):
 #   Column                                                         Non-Null Count  Dtype  
---  ------                                                         --------------  -----  
 0   Pays                                                           16047 non-null  object 
 1   Année                                                          16047 non-null  int64  
 2   Population totale                                              16047 non-null  int64  
 3   Produit                                                        16047 non-null  object 
 4   Alimentation pour touristes                                    16047 non-null  float64
 5   Aliments pour animaux                                          16047 non-null  float64
 6   Autres utilisations (non alimentaire)                          16047 non-null  float64
 7   Disponibilité alimentaire (Kcal/personne/jour)            

Conservation des champs pertinent pour l'analyse. Les autres seront supprimés.

In [ ]:
# Filter the dataframe for rows related to 'Poulets' (Chicken) and explore the available 'Élément' types
df_chicken = df_population_dispo_alimentaire[
    df_population_dispo_alimentaire['Produit'] == 'Viande de Volailles'
    ].drop(columns=['Produit', 'Année'])

# cols to keep depending on the criterias set up
cols_to_keep = ['Pays',
                'Alimentation pour touristes',
                'Population totale',
                'Disponibilité alimentaire (Kcal/personne/jour)',
                'Disponibilité de protéines en quantité (g/personne/jour)',
                'Disponibilité de matière grasse en quantité (g/personne/jour)',
                'Disponibilité alimentaire en quantité (kg/personne/an)',
                'Disponibilité intérieure',
                'Importations - Quantité',
                'Exportations - Quantité',
                'Nourriture',
                'Production',
                'Pertes',
                'Traitement']

df_chicken = df_chicken[cols_to_keep]
df_chicken.head()

,Pays,Alimentation pour touristes,Population totale,Disponibilité alimentaire (Kcal/personne/jour),Disponibilité de protéines en quantité (g/personne/jour),Disponibilité de matière grasse en quantité (g/personne/jour),Disponibilité alimentaire en quantité (kg/personne/an),Disponibilité intérieure,Importations - Quantité,Exportations - Quantité,Nourriture,Production,Pertes,Traitement
78,Afghanistan,0.0,36296113,5.0,0.54,0.33,1.53,57.0,29.0,0.0,55.0,28.0,2.0,0.0
173,Afrique du Sud,0.0,57009756,143.0,14.11,9.25,35.69,2118.0,514.0,63.0,2035.0,1667.0,83.0,0.0
265,Albanie,0.0,2884169,85.0,6.26,6.45,16.36,47.0,38.0,0.0,47.0,13.0,0.0,0.0
357,Algérie,0.0,41389189,22.0,1.97,1.50,6.38,277.0,2.0,0.0,264.0,275.0,13.0,0.0
452,Allemagne,0.0,82658409,71.0,7.96,4.16,19.47,1739.0,842.0,646.0,1609.0,1514.0,0.0,167.0


# **Création de champs personalisées : feature engineering**

Pour les besoins de l'étude, les indicateurs suivants ont été retenus:
- Ratio d'importations: Importations / (Importations + Production - Exportations - Pertes). Cet indicateur permet d'évaluer la part des importations dans l'approvisionnement total et d'identifier si un pays dépend majoritairement des importations
- Balance nette d'approvisionnement: (Importations - (Production + Exportations). Cet indicateur vise à mesurer le déficit ou l'excédent d'approvisionnement du pays
- Ratio de pertes sur production (Pertes / production). Cet indicateur exprime la proportion de pertes par rapport à la production nationale et, permet d'évaluer l'efficacité du contrôle des pertes, même lorsque la production est suffisante.

Pour le calcul de ces ratios, seul les valeurs positives ont été conservées, conformément à la logique métier et éviter toute incohérence analytique.

In [ ]:
import numpy as np

try:
    # Calcule le dénominateur pour le ratio d'importations, en s'assurant qu'il est positif.
    # Cela représente la quantité totale disponible après avoir tenu compte de la production, des importations, des exportations et des pertes.
    df_denominator = (
        (df_chicken['Production'] +
         df_chicken['Importations - Quantité'] -
         df_chicken['Pertes'] -
         df_chicken['Exportations - Quantité'])
    )

    # Calcule le 'Ratio importations / (importations + production - pertes - exportations)'
    # Ce ratio indique la dépendance aux importations pour l'approvisionnement total.
    # Gère la division par zéro en définissant le ratio à 0 si le dénominateur n'est pas positif.
    df_chicken['Ratio importations / (importations + production - pertes - exportations)'] = np.where(
        df_denominator > 0,
        df_chicken['Importations - Quantité'] / df_denominator,
        0 # Définit le ratio à 0 si le dénominateur est 0 ou négatif
    )

    # Calcule la 'Balance nette d'approvisionnement: Importations - (exportations + production)'
    # Cette métrique montre le solde net des importations par rapport à l'offre intérieure et aux exportations.
    df_chicken['Importations - (exportations + production)'] = df_chicken['Importations - Quantité'] \
    - (df_chicken['Exportations - Quantité'] + df_chicken['Production'])

    # Calcule le 'Ratio pertes / production'
    # Ce ratio mesure la proportion des pertes par rapport à la production nationale.
    # Gère la division par zéro en définissant le ratio à 0 si la production n'est pas positive.
    df_chicken['Ratio pertes / production'] = np.where(
        df_chicken['Production'] > 0,
        df_chicken['Pertes'] / df_chicken['Production'],
        0 # Définit le ratio à 0 si la production est 0
    )
except NameError:
    print("Erreur: Le DataFrame 'df_chicken' n'est pas défini. Veuillez vous assurer que les cellules précédentes, notamment celle qui filtre les données pour 'Viande de Volailles' (cellule db874c3d), ont été exécutées.")

# **Respect de critères importants**

Le nombre de pays couverts doit être au moins de 100 pays et couvrir au moins 60% de la population mondiale

In [ ]:
# See in % how our data cover the global population : > 60%
population_mondiale_2017 = 7615000000

population_mondiale = df_chicken['Population totale'].sum()

pourcentage_pop_mondiale = population_mondiale / population_mondiale_2017

nb_pays = df_chicken['Pays'].nunique()

print(f'Couverture actuelle des données comparés à la population mondiale en 2017 : {pourcentage_pop_mondiale:.2%}')
print(f'Nombre de pays couverts : {nb_pays}')

Couverture actuelle des données comparés à la population mondiale en 2017 : 96.77%
Nombre de pays couverts : 172


Nombre de pays couverts > 100 pays et 96% de la population mondiale couverte (> 60% de la population mondiale)

# **Génération du fichier CSV final**

In [ ]:
df_chicken.to_csv('chicken_country.csv', index=False)